# Ashworth et al. (2025), Poisson Gaussian-process fits with the Methods' settings, on dandiset 001708

**Paper.** Ashworth et al., "Multi-timescale dynamics organize descending pain modulation", bioRxiv
doi:10.64898/2025.12.15.693905, version 4, with its supplement; the claims file
(`claims/001708/10.64898-2025.12.15.693905.yaml`) was written from this text.

**Dataset.** DANDI:001708, version 0.260604.1023 (cited by the paper; the only published version), streamed.

**Why this notebook.** gp.ipynb fits the Gaussian processes with the code's settings (100 inducing
points, p_noise 0.01, 5000 iterations). The criterion's main reading is the Methods' 200 inducing
points and p_noise 0.001, with params.json's 10000 iterations. This notebook fits the same model
(rvm.fit_gp_fast, described in gp.ipynb) with the Methods' 200 inducing points and p_noise 0.001,
for 2500 iterations, with results also at 1250, to be compared with gp.ipynb's results at 2500
iterations (the code's settings). 10000 iterations are not run: on this machine one iteration over
the 82 cells takes about 0.55 s, so 10000 would take about 92 min, over the 60 min limit of a
notebook; 5000 would take about 46 min, which did not fit in the time left for this revision. Only the main inputs are fitted (ON and OFF 1500 s segments, evoked/ongoing NEUTRAL
cells over 960 s, ongoing NEUTRAL cells over their whole recordings), and only the claims that the
verifier asked to see under these settings are tested again: the pseudo-R2 tests, Table 1 and the
dispersion. The tests are the helper's, the same as in gp.ipynb.

| claim | tested here |
|---|---|
| fig5b-train-r2 | yes (variant settings) |
| fig5b-train-no-difference | yes (variant settings) |
| fig5b-test-r2 | yes (variant settings) |
| fig5b-test-group-differences | yes (variant settings) |
| table1-hyperparameters | yes (variant settings) |
| fig5c-dispersion | yes (variant settings) |
| table1-latent-variance-order | yes (variant settings) |
| fig5d-phase-profiles, figs2-phase-latency | in gp.ipynb only |

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import torch
import rvm
from rvm import signflip, r2_tests, group_tests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
T0 = time.time()
print("dandiset", rvm.DANDISET, "version", rvm.VERSION, "torch", torch.__version__)

## Data check

In [ ]:
sessions, cells = rvm.load_all()
print(f"loaded {len(sessions)} files in {time.time() - T0:.0f} s")
ong = cells[cells.protocol.isin(["ongoing", "evoked/ongoing"])].copy()
print(pd.crosstab(ong.protocol, ong.cls, margins=True))
print("subjects per class:", ong.groupby("cls").subject.nunique().to_dict(), "; files per class:", ong.groupby("cls").path.nunique().to_dict())
# Main inputs of the criterion (as gp.ipynb): ON and OFF 1500 s segments split at 1200 s; evoked/ongoing
# NEUTRAL cells, 960 s, split at 660 s; ongoing NEUTRAL cells over their whole recording less its last 300 s.
# Three batches of similar length, so that short series are not padded to the longest.
BATCH = {"seg": ([], []), "eo": ([], []), "whole": ([], [])}
for ci, c in ong.iterrows():
    tr = sessions[c.session]["trains"][c.unit]
    if c.protocol == "evoked/ongoing":
        BATCH["eo"][0].append(rvm.bin_split(tr, 960.0, 660.0)); BATCH["eo"][1].append(ci)
    elif c.cls == "NEUTRAL":
        BATCH["whole"][0].append(rvm.bin_split(tr, c.t_stop, c.t_stop - 300.0)); BATCH["whole"][1].append(ci)
    else:
        a, b, gap, buf = rvm.ongoing_segment(sessions[c.session], c.t_stop, 1500.0)
        BATCH["seg"][0].append(rvm.bin_split(rvm.cut(tr, a, b), 1500.0, 1200.0)); BATCH["seg"][1].append(ci)
for k, (xy, ids) in BATCH.items():
    print(f"batch {k}: {len(ids)} cells {cells.loc[ids, 'cls'].value_counts().to_dict()}; training bins {min(len(x[1]) for x in xy)} "
          f"to {max(len(x[1]) for x in xy)}, test bins {min(len(x[3]) for x in xy)} to {max(len(x[3]) for x in xy)}; "
          f"largest training count {max(int(x[1].max()) for x in xy)} (the contamination runs to 200)")

## Fits

200 inducing points and p_noise 0.001 (Methods, Gaussian process modeling, paragraph 2), every other
setting as in gp.ipynb, 2500 iterations with a snapshot at 1250.

In [ ]:
torch.set_num_threads(4)
IT2, IT = 1250, 2500
FIT = {}
for k, (xy, ids) in BATCH.items():
    t1 = time.time()
    FIT[k] = rvm.fit_gp_fast(xy, n_iter=IT, num_inducing=200, p_noise=0.001, lr=0.05, seed=0, snapshots=(IT2,))
    print(f"batch {k}, {len(ids)} cells: {time.time() - t1:.0f} s")

def table(it):
    rows = []
    for k, (xy, ids) in BATCH.items():
        r = FIT[k][it]
        for j, ci in enumerate(ids):
            rows.append(dict(ci=ci, batch=k, **{v: float(r[v][j]) for v in ["period", "lengthscale", "outputscale", "alpha", "train_r2", "test_r2"]}))
    d = pd.DataFrame(rows)
    d["cls"] = cells.loc[d.ci, "cls"].values
    d["subject"] = cells.loc[d.ci, "subject"].values
    d["file"] = cells.loc[d.ci, "path"].values
    return d

TAB = {it: table(it) for it in (IT2, IT)}
print("cells per class:", TAB[IT].cls.value_counts().to_dict())
print("non-finite values:", int((~np.isfinite(TAB[IT][["period", "lengthscale", "outputscale", "alpha", "train_r2", "test_r2"]])).sum().sum()))
for it, d in TAB.items():
    print(it, "iterations, class medians:", d.groupby("cls")[["train_r2", "test_r2", "period", "alpha", "outputscale"]].median().round(4).to_dict("index"))

## fig5b-train-r2

> A Poisson-type periodic Gaussian process fits the training data of all three cell classes with
> significant accuracy (pseudo-R2 different from 0; ON p = 0.0047, OFF p = 0.0004, NEUTRAL p < 0.0001).

In [ ]:
r2_tests(TAB[IT], "train_r2", "Methods' settings, 2500 iterations")
r2_tests(TAB[IT2], "train_r2", "Methods' settings, 1250 iterations")
print("paper (combined_data.csv) medians: ON 0.0448, OFF 0.0479, NEUTRAL 0.0281")

## fig5b-train-no-difference

> Training pseudo-R2 does not differ between the three classes (permuted, Bonferroni-corrected
> Mann-Whitney U tests; Kruskal-Wallis p = 0.117).

In [ ]:
group_tests(TAB[IT], "train_r2", "train pseudo-R2, Methods' settings, 2500 iterations")

## fig5b-test-r2 and fig5b-test-group-differences

> fig5b-test-r2: On held-out data ON cells generalize significantly (p = 0.0067), with most cells above
> 0; OFF cells as a population do not (p = 0.3127); NEUTRAL cells fall significantly below baseline (p = 0.0033).

> fig5b-test-group-differences: Test pseudo-R2 of NEUTRAL cells differs from that of ON cells (p <
> 0.001) and OFF cells (p = 0.006), ON and OFF cells do not differ (p = 0.939), and the Kruskal-Wallis
> test across the three is significant at test time (p < 0.001).

In [ ]:
r2_tests(TAB[IT], "test_r2", "Methods' settings, 2500 iterations")
print("paper (combined_data.csv) medians: ON 0.0372, OFF 0.0187, NEUTRAL -0.0128")

In [ ]:
group_tests(TAB[IT], "test_r2", "test pseudo-R2, Methods' settings, 2500 iterations")

## table1-hyperparameters, fig5c-dispersion, table1-latent-variance-order

> table1-hyperparameters: The fitted GP periods of ON and OFF cells cluster near 300 s (ON 297 ± 22 s,
> OFF 304 ± 27 s); NEUTRAL cells give 322 ± 54 s.

> fig5c-dispersion: ON and OFF cells have positive dispersion parameters (ON 0.63 ± 0.25, OFF 0.36 ±
> 0.53), NEUTRAL cells negative (-0.9 ± 0.63).

> table1-latent-variance-order: ON outputscale 17 ± 26, OFF 3.1 ± 4, NEUTRAL 0.1 ± 0.2.

In [ ]:
exp_p = {"ON": (297, 22), "OFF": (304, 27), "NEUTRAL": (322, 54)}
exp_l = {"ON": (0.73, 0.018), "OFF": (0.74, 0.013), "NEUTRAL": (0.74, 0.0068)}
for it in (IT, IT2):
    d = TAB[it]
    t = d.groupby("cls")[["period", "lengthscale", "outputscale", "alpha"]].agg(["mean", "std", "median"])
    print(f"--- {it} iterations"); print(t.round(3))
    for k in ["ON", "OFF", "NEUTRAL"]:
        m, l = t.loc[k, ("period", "mean")], t.loc[k, ("lengthscale", "mean")]
        print(f"{k}: period {m:.1f} vs {exp_p[k][0]} +- {exp_p[k][1]} (within: {abs(m - exp_p[k][0]) <= exp_p[k][1]}); "
              f"lengthscale {l:.3f} vs {exp_l[k][0]} +- {exp_l[k][1]} (within: {abs(l - exp_l[k][0]) <= exp_l[k][1]})")
    print("change of the period from its initial 300 s (median, min, max):",
          d.assign(ch=d.period - 300).groupby("cls").ch.agg(["median", "min", "max"]).round(1).to_dict("index"))

In [ ]:
d = TAB[IT]
for k, (m_, s_), seed in [("ON", (0.63, 0.25), 11), ("OFF", (0.36, 0.53), 12), ("NEUTRAL", (-0.9, 0.63), 13)]:
    dk = d[d.cls == k]
    o, p = signflip(dk, "alpha", "subject", seed=seed)
    o2, p2 = signflip(dk, "alpha", "file", seed=seed)
    print(f"{k}: alpha mean {dk.alpha.mean():.3f} +- {dk.alpha.std():.3f} (paper {m_} +- {s_}; within: {abs(dk.alpha.mean() - m_) <= s_}); "
          f"median {o:.3f}; sign-flip p by animal {p:.4f}, by file {p2:.4f}")

In [ ]:
for it in (IT, IT2):
    g = TAB[it].groupby("cls").outputscale.agg(["mean", "std", "median"])
    print(f"{it} iterations: outputscale {g.round(3).to_dict('index')}; order of means "
          f"{g['mean'].sort_values(ascending=False).index.tolist()}, of medians {g['median'].sort_values(ascending=False).index.tolist()}")
for k, (m_, s_) in {"ON": (17, 26), "OFF": (3.1, 4), "NEUTRAL": (0.1, 0.2)}.items():
    v = TAB[IT].query("cls == @k").outputscale.mean()
    print(f"{k}: mean {v:.3f} vs {m_} +- {s_}: within {abs(v - m_) <= s_}")

## Summary

With the Methods' 200 inducing points and p_noise 0.001, at 2500 iterations (a variant; the outcomes are those of
gp.ipynb, whose settings match the code's stored fits):

| claim | under these settings |
|---|---|
| fig5b-train-r2 | holds: medians 0.0556, 0.0566, 0.0219; p by animal 0.0170, 0.0040, 0.0001 |
| fig5b-train-no-difference | fails: ON against NEUTRAL Bonferroni p 0.006 (by file), 0.030 (by animal); Kruskal-Wallis 0.009 and 0.027 |
| fig5b-test-r2 | holds: ON median 0.0379, p 0.0079, 20 of 25 above 0; OFF p 0.373; NEUTRAL median -0.0085, p 0.046 |
| fig5b-test-group-differences | partial: NEUTRAL against ON below 0.001, against OFF 0.054 (by file); ON against OFF 0.79; Kruskal-Wallis 0.001 |
| table1-hyperparameters | holds: periods 296.3, 305.0, 313.4 s; lengthscales 0.724, 0.735, 0.740 |
| fig5c-dispersion | holds: alpha 0.656, 0.367, -0.768; p by animal 0.0045, 0.0168, below 0.0001 |
| table1-latent-variance-order | holds: means 22.5, 4.31, 0.118, in order and within the spread |

The ON mean outputscale rose from 17.9 at 1250 to 22.5 at 2500 iterations, so it is not settled. 10000 iterations
were not run (about 92 min, over the notebook limit).


In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min")